In [1]:
from pathlib import Path
import subprocess


def run_ffmpeg(*args: str) -> None:
    subprocess.run(
        ["ffmpeg", "-hide_banner", "-y", *map(str, args)],
        check=True,
    )


def rotate_video(input_path: str, output_path: str, degrees: int) -> None:
    filters = {
        90: "transpose=clock",
        180: "hflip,vflip",
        270: "transpose=cclock",
    }
    if degrees not in filters:
        raise ValueError("degrees must be 90, 180, or 270")

    run_ffmpeg(
        "-i", input_path,
        "-vf", filters[degrees],
        "-c:a", "copy",
        output_path,
    )


def crop_video(
    input_path: str,
    output_path: str,
    width: int,
    height: int,
    x: int,
    y: int,
) -> None:
    run_ffmpeg(
        "-i", input_path,
        "-vf", f"crop={width}:{height}:{x}:{y}",
        "-c:a", "copy",
        output_path,
    )


def change_speed(input_path: str, output_path: str, speed: float) -> None:
    if speed <= 0:
        raise ValueError("speed must be positive")

    # 영상은 속도 배율만큼 PTS를 줄임.
    video_filter = f"setpts=PTS/{speed}"

    # atempo는 0.5~2.0만 지원하므로 범위를 벗어나면 여러 필터로 나눔.
    audio_filters = []
    remaining = speed
    while remaining > 2:
        audio_filters.append("atempo=2")
        remaining /= 2
    while remaining < 0.5:
        audio_filters.append("atempo=0.5")
        remaining /= 0.5
    audio_filters.append(f"atempo={remaining}")

    run_ffmpeg(
        "-i", input_path,
        "-filter:v", video_filter,
        "-filter:a", ",".join(audio_filters),
        output_path,
    )


def extract_frames(
    input_path: str,
    output_dir: str,
    fps: float | None = None,
    image_format: str = "png",
) -> None:
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)

    args = ["-i", input_path]
    if fps is not None:
        args += ["-vf", f"fps={fps}"]

    args += [str(output / f"%08d.{image_format}")]
    run_ffmpeg(*args)


def frames_to_video(
    frames_dir: str,
    output_path: str,
    fps: float,
    audio_source: str | None = None,
) -> None:
    args = [
        "-framerate", str(fps),
        "-i", str(Path(frames_dir) / "%08d.png"),
    ]

    if audio_source:
        args += [
            "-i", audio_source,
            "-map", "0:v:0",
            "-map", "1:a?",
            "-c:a", "copy",
            "-shortest",
        ]

    args += [
        "-c:v", "libx264",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        output_path,
    ]
    run_ffmpeg(*args)

```python
input_video = "/Users/hyeon/Videos/input.mp4"
```

### 90도 시계 방향 회전
```python
rotate_video(input_video, "/Users/hyeon/Videos/rotated.mp4", 90)
```

### 좌상단 기준 x=100, y=50에서 720×1280 영역만 남김
```python
crop_video(input_video, "/Users/hyeon/Videos/cropped.mp4", 720, 1280, 100, 50)
```

### 2배 빠르게, 오디오도 함께 변경
```python
change_speed(input_video, "/Users/hyeon/Videos/fast.mp4", 2.0)
```

### 원본 FPS 그대로 모든 프레임 PNG 저장
```python
extract_frames(
    input_video,
    "/Users/hyeon/SeSAC_Project/Sapiens2/inputs/video_frames",
)
```

### 초당 10프레임만 추출: HPE 테스트를 빠르게 할 때 유용
```python
extract_frames(
    input_video,
    "/Users/hyeon/SeSAC_Project/Sapiens2/inputs/video_frames_10fps",
    fps=10,
)
```

### HPE 결과 프레임을 30fps 영상으로 조립하고 원본 오디오 유지
```python
frames_to_video(
    "/Users/hyeon/SeSAC_Project/Sapiens2/outputs/body18_frames",
    "/Users/hyeon/Videos/input_body18_hpe.mp4",
    fps=30,
    audio_source=input_video,
)
```

In [6]:
input_video = "/Users/hyeon/SeSAC_Project/Sapiens2/inputs/videos/test_video_1.mp4"

In [12]:
rotated = "/Users/hyeon/SeSAC_Project/Sapiens2/inputs/videos/test1/rotated.mp4"

In [ ]:
rotate_video(input_video, "/Users/hyeon/SeSAC_Project/Sapiens2/inputs/videos/rotated.mp4", 90)

Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/Users/hyeon/SeSAC_Project/Sapiens2/inputs/videos/test_video_1.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 1
    compatible_brands: isomavc1mp42
    creation_time   : 2023-08-05T05:17:39.000000Z
  Duration: 00:00:20.00, start: 0.000000, bitrate: 7929 kb/s
  Stream #0:0[0x1](und): Video: h264 (Constrained Baseline) (avc1 / 0x31637661), yuv420p(tv, bt709, progressive), 960x540 [SAR 1:1 DAR 16:9], 7928 kb/s, 25 fps, 25 tbr, 25 tbn (default)
Stream mapping:
  Stream #0:0 -> #0:0 (h264 (native) -> h264 (libx264))
Press [q] to stop, [?] for help
[libx264 @ 0xc3482ca80] using SAR=1/1
[libx264 @ 0xc3482ca80] using cpu capabilities: ARMv8 NEON DotProd I8MM
[libx264 @ 0xc3482ca80] profile High, level 3.1, 4:2:0, 8-bit
[libx264 @ 0xc3482ca80] 264 - core 165 r3222 b35605a - H.264/MPEG-4 AVC codec - Copyleft 2003-2025 - http://www.videolan.org/x264.html - options: cabac=1 ref=3 deblock=1:0:0 analyse=0x3:0x113 me=hex subme=7 psy=1 psy_r

In [13]:
extract_frames(
    rotated,
    "/Users/hyeon/SeSAC_Project/Sapiens2/inputs/videos/test1/frames",
)

Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/Users/hyeon/SeSAC_Project/Sapiens2/inputs/videos/test1/rotated.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2avc1mp41
    encoder         : Lavf63.1.101
  Duration: 00:00:20.00, start: 0.000000, bitrate: 4443 kb/s
  Stream #0:0[0x1](und): Video: h264 (High) (avc1 / 0x31637661), yuv420p(tv, bt709, progressive), 540x960 [SAR 1:1 DAR 9:16], 4440 kb/s, 25 fps, 25 tbr, 12800 tbn (default)
    Metadata:
      handler_name    : VideoHandler
      encoder         : Lavc63.1.101 libx264
Stream mapping:
  Stream #0:0 -> #0:0 (h264 (native) -> png (native))
Press [q] to stop, [?] for help
[swscaler @ 0xa4dfc8000] No accelerated colorspace conversion found from yuv420p to rgb24.
[swscaler @ 0xa4dfc8000] [swscaler @ 0xa4dfd8000] No accelerated colorspace conversion found from yuv420p to rgb24.
[swscaler @ 0xa4dfc8000] [swscaler @ 0xa4dfe8000] No accelerated colorspace conversion found from yuv420p to r

In [16]:
frames_to_video(
    "/Users/hyeon/SeSAC_Project/Sapiens2/outputs/test1_coco17_frames",
    "/Users/hyeon/SeSAC_Project/Sapiens2/outputs/test1/test1_result.mp4",
    fps=25,
    audio_source=rotated,
)

Input #0, image2, from '/Users/hyeon/SeSAC_Project/Sapiens2/outputs/test1_coco17_frames/%08d.png':
  Duration: 00:00:20.00, start: 0.000000, bitrate: N/A
  Stream #0:0: Video: png, rgb24(pc, gbr/unknown/unknown), 540x960, 25 fps, 25 tbr, 25 tbn
Input #1, mov,mp4,m4a,3gp,3g2,mj2, from '/Users/hyeon/SeSAC_Project/Sapiens2/inputs/videos/test1/rotated.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2avc1mp41
    encoder         : Lavf63.1.101
  Duration: 00:00:20.00, start: 0.000000, bitrate: 4443 kb/s
  Stream #1:0[0x1](und): Video: h264 (High) (avc1 / 0x31637661), yuv420p(tv, bt709, progressive), 540x960 [SAR 1:1 DAR 9:16], 4440 kb/s, 25 fps, 25 tbr, 12800 tbn (default)
    Metadata:
      handler_name    : VideoHandler
      encoder         : Lavc63.1.101 libx264
Stream mapping:
  Stream #0:0 -> #0:0 (png (native) -> h264 (libx264))
Press [q] to stop, [?] for help
[libx264 @ 0xb13021180] using cpu capabilities: ARMv8 NEON DotProd I8MM